# RNN and LSTM experiments

Baseline RNNs and LSTMs on the Bulgarian emotion corpus. Data not included: see `data/README.md`.

### Preprocessing dataset

In [1]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

tf.test.is_gpu_available()

True

In [18]:
! pip install -U imbalanced-learn

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import os

# Configuration
VOCAB_SIZE = 20000
MAX_LEN = 27
EMBEDDING_DIM = 128
EPOCHS = 20
BATCH_SIZE = 512

# Define constants for file paths
DATA_DIR = "data"
MODEL_DIR = "models/lstm"
TUNER_DIR = "models/lstm/tuner"
REPORTS_DIR = "reports/lstm"
PLOTS_DIR = "plots/lstm"

# Create directories if they don't exist
for d in [DATA_DIR, MODEL_DIR, TUNER_DIR, REPORTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Data File Paths ---
TRAIN_PKL = "FINAL_but_cooler.pkl"
TEST_CSV = "NLP_features_engineered_but_even_cooler.csv"

# --- Output File Paths ---
BASELINE_LSTM_PATH = os.path.join(MODEL_DIR, "baseline_lstm.keras")
STACKED_LSTM_PATH = os.path.join(MODEL_DIR, "stacked_lstm.keras")
BIDIR_LSTM_PATH = os.path.join(MODEL_DIR, "bidir_lstm.keras")
LSTM_DROPOUT_PATH = os.path.join(MODEL_DIR, "lstm_dropout.keras")
LSTM_L2_PATH = os.path.join(MODEL_DIR, "lstm_l2.keras")
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "final_lstm_model.keras")

FINAL_RESULTS_CSV = os.path.join(REPORTS_DIR, "final_model_performance.csv")
TUNING_SUMMARY_CSV = os.path.join(REPORTS_DIR, "tuning_summary.csv")
BEST_CONFIG_FILE = os.path.join(REPORTS_DIR, "best_hyperparameters.json")
COMPARISON_TXT = os.path.join(REPORTS_DIR, "model_comparison.txt")
FINAL_REPORT_TXT = os.path.join(REPORTS_DIR, "final_classification_report.txt")

# --- Plot File Paths ---
FINAL_HISTORY_PLOT = os.path.join(PLOTS_DIR, "final_training_history.png")
FINAL_CM_PLOT = os.path.join(PLOTS_DIR, "final_confusion_matrix.png")


# Data Preparation
# Load the datasets
df_train = pd.read_pickle(TRAIN_PKL)
df_test = pd.read_csv(TEST_CSV)

# Clean up columns
df_train = df_train[['text', 'emotion']]
df_test = df_test[['Sentence', 'Read emotions']]
df_test.rename(columns={'Sentence': 'text', 'Read emotions': 'emotion'}, inplace=True)

# Undersampling the major class
# Find the count of the second-largest class
n_target = df_train['emotion'].value_counts().iloc[1]

# Separate the majority class from the rest
df_neutral = df_train[df_train['emotion'] == 'neutral']
df_others = df_train[df_train['emotion'] != 'neutral']

# Randomly sample the majority class down to the target size
df_neutral_undersampled = df_neutral.sample(n=n_target, random_state=42)

# Combine the undersampled majority class with the other classes
df_train = pd.concat([df_neutral_undersampled, df_others])

# Shuffle the dataset to mix the rows
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print("Value counts after undersampling:")
print(df_train['emotion'].value_counts())

# Encode emotion labels
label_encoder = LabelEncoder()
df_train['emotion_encoded'] = label_encoder.fit_transform(df_train['emotion'])
df_test['emotion_encoded'] = label_encoder.transform(df_test['emotion'])

# Get the mapping for reference
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)
NUM_CLASSES = len(label_mapping)

# Create train/validation split from the new balanced training data
X = df_train['text']
y = df_train['emotion_encoded']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Prepare the final test set
X_test = df_test['text']
y_test = df_test['emotion_encoded']


# --- 4. TOKENIZE & PAD TEXT ---
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding='post', truncating='post')


## Simple LSTM Model (Baseline)

In [30]:
# Build the model
model_baseline_lstm = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dense(NUM_CLASSES, activation='softmax')
])

model_baseline_lstm.compile(optimizer='adam',
                            loss='sparse_categorical_crossentropy',
                            metrics=['accuracy'])

model_baseline_lstm.summary()

In [31]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_baseline_lstm = model_baseline_lstm.fit(X_train_pad, y_train,
                                                epochs=EPOCHS,
                                                batch_size=BATCH_SIZE,
                                                validation_data=(X_val_pad, y_val),
                                                callbacks=[early_stopping],
                                                verbose=1)

model_baseline_lstm.save(BASELINE_LSTM_PATH)

In [32]:
loss, accuracy = model_baseline_lstm.evaluate(X_test_pad, y_test)
print(f"Baseline LSTM Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = model_baseline_lstm.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

## Stacked LSTM Model

In [33]:
model_stacked_lstm = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(NUM_CLASSES, activation='softmax')
])

model_stacked_lstm.compile(optimizer='adam',
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy'])

model_stacked_lstm.summary()

In [34]:
history_stacked_lstm = model_stacked_lstm.fit(X_train_pad, y_train,
                                              epochs=EPOCHS,
                                              batch_size=BATCH_SIZE,
                                              validation_data=(X_val_pad, y_val),
                                              callbacks=[early_stopping],
                                              verbose=1)

model_stacked_lstm.save(STACKED_LSTM_PATH)

In [35]:
loss, accuracy = model_stacked_lstm.evaluate(X_test_pad, y_test)
print(f"Stacked LSTM Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = model_stacked_lstm.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

## Bidirectional LSTM Model

In [36]:
model_bidir_lstm = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64)),
    Dense(NUM_CLASSES, activation='softmax')
])

model_bidir_lstm.compile(optimizer='adam',
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy'])

model_bidir_lstm.summary()

In [37]:
history_bidir_lstm = model_bidir_lstm.fit(X_train_pad, y_train,
                                            epochs=EPOCHS,
                                            batch_size=BATCH_SIZE,
                                            validation_data=(X_val_pad, y_val),
                                            callbacks=[early_stopping],
                                            verbose=1)

model_bidir_lstm.save(BIDIR_LSTM_PATH)

In [38]:
loss, accuracy = model_bidir_lstm.evaluate(X_test_pad, y_test)
print(f"Bidirectional LSTM Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = model_bidir_lstm.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

## LSTM Model with Dropout

In [39]:
model_lstm_dropout = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model_lstm_dropout.compile(optimizer='adam',
                             loss='sparse_categorical_crossentropy',
                             metrics=['accuracy'])

model_lstm_dropout.summary()

In [40]:
history_lstm_dropout = model_lstm_dropout.fit(X_train_pad, y_train,
                                                epochs=EPOCHS,
                                                batch_size=BATCH_SIZE,
                                                validation_data=(X_val_pad, y_val),
                                                callbacks=[early_stopping],
                                                verbose=1)

model_lstm_dropout.save(LSTM_DROPOUT_PATH)

In [41]:
loss, accuracy = model_lstm_dropout.evaluate(X_test_pad, y_test)
print(f"LSTM with Dropout Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = model_lstm_dropout.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

## LSTM with L2 regularizer

In [48]:
from tensorflow.keras.regularizers import l2

model_lstm_l2 = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    LSTM(32, kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model_lstm_l2.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

model_lstm_l2.summary()

In [49]:
history_lstm_l2 = model_lstm_l2.fit(X_train_pad, y_train,
                                      epochs=EPOCHS,
                                      batch_size=BATCH_SIZE,
                                      validation_data=(X_val_pad, y_val),
                                      callbacks=[early_stopping],
                                      verbose=1)

model_lstm_l2.save(LSTM_L2_PATH)

In [50]:
loss, accuracy = model_lstm_l2.evaluate(X_test_pad, y_test)
print(f"LSTM with L2 Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = model_lstm_l2.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

## Hyperparameter Tuning with Keras Tuner

In [51]:
!pip install keras-tuner -q

In [52]:
import keras_tuner as kt
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam

In [53]:
def build_model(hp):
    model = Sequential()
    model.add(Embedding(input_dim=VOCAB_SIZE, 
                        output_dim=hp.Int('embedding_dim', min_value=64, max_value=256, step=64),
                        input_length=MAX_LEN))
    
    # Tune number of LSTM layers
    for i in range(hp.Int('num_lstm_layers', 1, 2)):
        model.add(Bidirectional(
            LSTM(
                units=hp.Int(f'lstm_units_{i}', min_value=32, max_value=128, step=32),
                return_sequences=True if i < hp.Int('num_lstm_layers', 1, 2) - 1 else False,
                kernel_regularizer=l2(hp.Float('l2', min_value=1e-4, max_value=1e-2, sampling='log'))
            )
        ))
        model.add(Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)))
    
    model.add(Dense(NUM_CLASSES, activation='softmax'))
    
    model.compile(
        optimizer=Adam(hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [54]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,  # Number of different hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial
    directory=TUNER_DIR,
    project_name='emotion_lstm_tuning'
)

In [55]:
tuner.search(X_train_pad, y_train,
             epochs=5, # Short epochs for quick tuning
             validation_data=(X_val_pad, y_val),
             callbacks=[early_stopping])

In [56]:
tuner.results_summary()

## Final Model Training & Evaluation

In [57]:
# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:")
print(best_hps.values)

In [58]:
# Build the model with the best hyperparameters
final_model = tuner.hypermodel.build(best_hps)
final_model.summary()

In [59]:
# Train the final model on the full training data (train + validation)
X_full_train = np.concatenate((X_train_pad, X_val_pad))
y_full_train = np.concatenate((y_train, y_val))

history_final = final_model.fit(X_full_train, y_full_train,
                                epochs=25, # Train for longer
                                batch_size=BATCH_SIZE,
                                validation_data=(X_val_pad, y_val), # Still use validation for monitoring
                                callbacks=[early_stopping],
                                verbose=1)

final_model.save(FINAL_MODEL_PATH)

In [60]:
print("--- Final Model Evaluation on Test Data ---")
loss, accuracy = final_model.evaluate(X_test_pad, y_test)
print(f"Final Model Test Accuracy: {accuracy*100:.2f}%")

y_pred_probs = final_model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
report = classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_)
print(report)

with open(FINAL_REPORT_TXT, 'w') as f:
    f.write(report)

## Plotting and Reporting

In [61]:
import matplotlib.pyplot as plt

pd.DataFrame(history_final.history).plot(figsize=(10, 7))
plt.title("Final Model Training History")
plt.grid(True)
plt.savefig(FINAL_HISTORY_PLOT)
plt.show()

In [62]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import json

In [63]:
report_dict = classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_, output_dict=True)
final_results = {
    "val_accuracy": history_final.history['val_accuracy'][-1],
    "val_loss": history_final.history['val_loss'][-1],
    "test_accuracy": report_dict['accuracy'],
    "test_f1_macro": report_dict['macro avg']['f1-score'],
    "test_precision_macro": report_dict['macro avg']['precision'],
    "test_recall_macro": report_dict['macro avg']['recall']
}
pd.DataFrame([final_results]).to_csv(FINAL_RESULTS_CSV, index=False)

# Save best HPs to a file
with open(BEST_CONFIG_FILE, 'w') as f:
    json.dump(best_hps.values, f, indent=4)

In [64]:
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Final Model Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig(FINAL_CM_PLOT)
plt.show()

## Compare All Models

In [65]:
!pip install tabulate

In [66]:
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score, accuracy_score
from tabulate import tabulate
import json

In [67]:
model_paths = {
    "Baseline LSTM": BASELINE_LSTM_PATH,
    "Stacked LSTM": STACKED_LSTM_PATH,
    "Bidirectional LSTM": BIDIR_LSTM_PATH,
    "LSTM + Dropout": LSTM_DROPOUT_PATH,
    "LSTM + L2": LSTM_L2_PATH,
    "Tuned Final LSTM": FINAL_MODEL_PATH
}

results = []

In [68]:
for name, path in model_paths.items():
    print(f"Evaluating {name}...")
    model = load_model(path)
    
    # Evaluate on Validation Set
    val_preds_probs = model.predict(X_val_pad, verbose=0)
    val_preds = np.argmax(val_preds_probs, axis=1)
    val_f1 = f1_score(y_val, val_preds, average='macro')
    val_acc = accuracy_score(y_val, val_preds)
    
    # Evaluate on Test Set
    test_preds_probs = model.predict(X_test_pad, verbose=0)
    test_preds = np.argmax(test_preds_probs, axis=1)
    test_f1 = f1_score(y_test, test_preds, average='macro')
    test_acc = accuracy_score(y_test, test_preds)
    
    results.append({
        "Model": name,
        "Val F1 (Macro)": val_f1,
        "Val Accuracy": val_acc,
        "Test F1 (Macro)": test_f1,
        "Test Accuracy": test_acc,
        "Num Parameters": model.count_params()
    })

comparison_df = pd.DataFrame(results).sort_values(by="Test F1 (Macro)", ascending=False)

In [69]:
# Get all tuning results
all_trials = []
for trial in tuner.oracle.get_best_trials(num_trials=tuner.oracle.max_trials):
    trial_data = trial.hyperparameters.values.copy()
    trial_data['score'] = trial.score
    all_trials.append(trial_data)

tuning_df = pd.DataFrame(all_trials).sort_values(by='score', ascending=False)
tuning_df.to_csv(TUNING_SUMMARY_CSV, index=False)

In [70]:
# Save comparison as a clean text file
report_str = "Model Comparison Report\n"
report_str += "="*80 + "\n"
report_str += tabulate(comparison_df, headers='keys', tablefmt='grid', floatfmt=".4f")
report_str += "\n\nBest Hyperparameters (from Tuner)\n"
report_str += "="*80 + "\n"
report_str += json.dumps(best_hps.values, indent=4)
report_str += "\n\nFinal Model Classification Report (on Test Data)\n"
report_str += "="*80 + "\n"
report_str += report

with open(COMPARISON_TXT, 'w') as f:
    f.write(report_str)

print(report_str)

In [71]:
print("All artifacts saved successfully!")
print("-"*80)
print(f"✓ {FINAL_MODEL_PATH} - Final trained model")
print(f"✓ {FINAL_RESULTS_CSV} - Performance comparison")
print(f"✓ {TUNING_SUMMARY_CSV} - All tested configurations")
print(f"✓ {BEST_CONFIG_FILE} - Best hyperparameters")
print(f"✓ {COMPARISON_TXT} - Detailed comparison report")
print(f"✓ {FINAL_HISTORY_PLOT} - Training curves")
print(f"✓ {FINAL_CM_PLOT} - Confusion matrix")
print(f"✓ {FINAL_REPORT_TXT} - Detailed metrics")
print(f"✓ {TUNER_DIR}/ - All tuning checkpoints")
print("-"*80)

print("\nFINAL RESULTS:")
print("-"*80)
print(f"Best Validation F1: {final_results['val_f1_macro']:.4f}")
print(f"Best Test F1: {final_results['test_f1_macro']:.4f}")
print(f"Test Accuracy: {final_results['test_accuracy']:.4f}")
print("-"*80)